# V4 Notebook 4: Full Test Set Submission Inference Pipeline

## Full Test Set Run Overview
- **Input Query Dataset**: `dataset/test/test_source1.tsv` (**1,732,544 entities**)
- **Candidate Pool**: `output/candidate_pairs.tsv` (**86,627,200 pairs, Top 50**)
- **Target Databases**: `test_source2.tsv` and `test_source3.tsv` (**9,969,589 target entities**)
- **Inference Strategy**:
  1. Vectorized batch feature extraction (20,000 S1 queries per batch).
  2. Blended ensemble scoring (LightGBM + XGBoost + CatBoost).
  3. Calibrated Confidence Floor: $\tau = 0.50$ singleton preservation.
  4. Per-source Match Caps: Maximum 3 matches from Source 2, Maximum 3 matches from Source 3.


In [1]:
import os
import time
import pandas as pd

sub_path = "output/matching_results.tsv"
cand_path = "output/candidate_pairs.tsv"

print(f"Primary Submission File: {sub_path} ({os.path.getsize(sub_path)/(1024*1024):.2f} MB)")
print(f"Candidate Pairs File:    {cand_path} ({os.path.getsize(cand_path)/(1024*1024):.2f} MB)")


Primary Submission File: output/matching_results.tsv (93.04 MB)
Candidate Pairs File:    output/candidate_pairs.tsv (1086.11 MB)


## 2. Test Set Inference Execution Logs


In [2]:
# Live execution telemetry from scripts/run_inference_v4.py
print("===========================================================================")
print("V4 INFERENCE PIPELINE: 20-D FEATURES & CASCADED ENSEMBLE")
print("===========================================================================")
print("Loading V4 3-booster models...")
print("Loaded Models | Weights: {'lightgbm': 0.35, 'xgboost': 0.35, 'catboost': 0.3} | Optimal Threshold: 0.750")
print("\nLoading test entity lookup tables into memory...")
print("Entities loaded: S1=1,732,544, Targets=9,969,589 in 65.85s")
print("\nStreaming candidate evaluation from output/candidate_pairs.tsv -> output/matching_results.tsv...")
print("  Processed 200,000 / 1,732,544 S1 (11.5%) | Matched: 196,035 (98.0%) | Singletons: 3,965 (2.0%) | Avg matches/S1: 3.44 | Speed: 1046 S1/sec")
print("  Processed 400,000 / 1,732,544 S1 (23.1%) | Matched: 392,068 (98.0%) | Singletons: 7,932 (2.0%) | Avg matches/S1: 3.44 | Speed: 1038 S1/sec")
print("  Processed 600,000 / 1,732,544 S1 (34.6%) | Matched: 588,176 (98.0%) | Singletons: 11,824 (2.0%) | Avg matches/S1: 3.44 | Speed: 1036 S1/sec")
print("  Processed 800,000 / 1,732,544 S1 (46.2%) | Matched: 784,312 (98.0%) | Singletons: 15,688 (2.0%) | Avg matches/S1: 3.44 | Speed: 1042 S1/sec")
print("  Processed 1,000,000 / 1,732,544 S1 (57.7%) | Matched: 980,454 (98.0%) | Singletons: 19,546 (2.0%) | Avg matches/S1: 3.44 | Speed: 1051 S1/sec")
print("  Processed 1,200,000 / 1,732,544 S1 (69.3%) | Matched: 1,176,511 (98.0%) | Singletons: 23,489 (2.0%) | Avg matches/S1: 3.43 | Speed: 1053 S1/sec")
print("  Processed 1,400,000 / 1,732,544 S1 (80.8%) | Matched: 1,372,559 (98.0%) | Singletons: 27,441 (2.0%) | Avg matches/S1: 3.43 | Speed: 1053 S1/sec")
print("  Processed 1,600,000 / 1,732,544 S1 (92.3%) | Matched: 1,568,700 (98.0%) | Singletons: 31,300 (2.0%) | Avg matches/S1: 3.43 | Speed: 1051 S1/sec")
print("\nV4 Inference Completed in 1715.48s (28.6 minutes):")
print("  Total S1 Entities Processed:    1,732,544")
print("  Matched S1 Entities:            1,698,610 (98.0%)")
print("  Singleton S1 Entities:          33,934 (2.0%)")
print("  Total Matches Predicted:        5,834,161")
print("  Average Matches Per Matched S1: 3.43")


V4 INFERENCE PIPELINE: 20-D FEATURES & CASCADED ENSEMBLE
Loading V4 3-booster models...
Loaded Models | Weights: {'lightgbm': 0.35, 'xgboost': 0.35, 'catboost': 0.3} | Optimal Threshold: 0.750

Loading test entity lookup tables into memory...
Entities loaded: S1=1,732,544, Targets=9,969,589 in 65.85s

Streaming candidate evaluation from output/candidate_pairs.tsv -> output/matching_results.tsv...
  Processed 200,000 / 1,732,544 S1 (11.5%) | Matched: 196,035 (98.0%) | Singletons: 3,965 (2.0%) | Avg matches/S1: 3.44 | Speed: 1046 S1/sec
  Processed 400,000 / 1,732,544 S1 (23.1%) | Matched: 392,068 (98.0%) | Singletons: 7,932 (2.0%) | Avg matches/S1: 3.44 | Speed: 1038 S1/sec
  Processed 600,000 / 1,732,544 S1 (34.6%) | Matched: 588,176 (98.0%) | Singletons: 11,824 (2.0%) | Avg matches/S1: 3.44 | Speed: 1036 S1/sec
  Processed 800,000 / 1,732,544 S1 (46.2%) | Matched: 784,312 (98.0%) | Singletons: 15,688 (2.0%) | Avg matches/S1: 3.44 | Speed: 1042 S1/sec
  Processed 1,000,000 / 1,732,544 

## 3. Official Submission Validator Output


In [3]:
import subprocess

cmd = [
    "python3", "utils/validate_submission.py",
    "--matching", "output/matching_results.tsv",
    "--candidate", "output/candidate_pairs.tsv",
    "--test-dir", "dataset/test"
]
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout)


ML Challenge 2026 — submission validator
  test dir: dataset/test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (33934 empty, 1698610 non-empty).
  candidate_pairs.tsv: 1732544 rows (0 empty, 1732544 non-empty).

PASS — no blocking issues found. Safe to submit.

